# Module 9: File Handling

**Utrains Python Fundamentals** &middot; lab notebook

Part 5 of the course: Working with the Outside World.

## What you will be able to do by the end

- Open a file in the right mode for what you are about to do
- Prefer with open(...) so files always get closed
- Read and write JSON, the format every API speaks
- Locate a config file on disk with pathlib

## How to use this notebook

Run every cell in order with **Shift + Enter**. Read the markdown before each
block, then run the code and compare what you see against what you expected.

Two cells in this notebook are marked **Your turn**. They contain `____` where
a piece of the syntax is missing. They will fail if you run them as they are.
That is deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**. It is a short task with no code written for
you, so you have to put the module together yourself.

File handling lets your program read and write data outside of memory, so it
survives after the script finishes running.

## Setting up a scratch folder

Everything in this notebook writes into a `scratch/` folder next to it, so
nothing else on your machine is touched. Run this cell first.

In [ ]:
from pathlib import Path

WORK = Path("scratch")
WORK.mkdir(exist_ok=True)

print("writing files into:", WORK.resolve())

## File modes

You choose a mode when you open a file, which controls what you may do with it.

- `"r"` read, and the default
- `"w"` write, and **overwrites** anything already there
- `"a"` append, adding to the end
- `"x"` create, and fails if the file already exists
- `"rb"` and `"wb"` the same as r and w, but for binary data such as images

## Opening, writing and reading

The long form uses `open()` and `close()` as a pair.

In [ ]:
file = open(WORK / "sample.txt", "w")
file.write("Hello, this is a test file.")
file.close()

file = open(WORK / "sample.txt", "r")
content = file.read()
print(content)
file.close()

## The with statement

Opening a file with `with` closes it for you automatically, even if an error
happens partway through. This is the pattern you should reach for by default.

> If you forget to close a file it can stay locked or lose unsaved data. `with`
> removes that risk entirely.

In [ ]:
with open(WORK / "output.txt", "w") as file:
    file.writelines(["Line 1\n", "Line 2\n", "Line 3\n"])

with open(WORK / "output.txt", "a") as file:
    file.write("This line will be appended.\n")

with open(WORK / "output.txt", "r") as file:
    print(file.read())

# Reading one line at a time, or all lines into a list.
with open(WORK / "output.txt") as file:
    print("first line:", repr(file.readline()))

with open(WORK / "output.txt") as file:
    print("as a list :", file.readlines())

## Reading a file that ships with the repo

`data/servers.txt` sits alongside this notebook's folder. Reading a real file
from disk is the same code, just a different path.

In [ ]:
servers_file = Path("..") / "data" / "servers.txt"

with open(servers_file) as f:
    for line in f:
        print("-", line.strip())

---

### Your turn 1

Append a new incident summary to a running log file, then read the whole log back. Pick the mode that adds without wiping the file.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
log_path = WORK / "incidents.log"

with open(log_path, "a") as f:
    f.write("INC-4412 database connection failures resolved\n")

with open(log_path, "r") as f:
    print(f.read())

Run that cell two or three times. The log grows each time, which is exactly
what append mode is for. Change the mode to `"w"` and run it again to watch the
earlier lines disappear.

## Checking and deleting files

Checking whether a file exists and deleting one both live in the `os` module,
which ships with Python.

In [ ]:
import os

target = WORK / "sample.txt"

if os.path.exists(target):
    print("File exists")
    os.remove(target)
    print("and now it does not:", os.path.exists(target))
else:
    print("File not found")

## Working with JSON

JSON is a text format almost every API uses. It maps directly onto a Python
dictionary. `json.dumps` turns a Python object into a JSON **string**, and
`json.loads` turns a JSON string back into a Python object. The `s` stands for
string; without it, `json.dump` and `json.load` work on **files** instead.

In [ ]:
import json

person = {"name": "Alice", "age": 25}

as_text = json.dumps(person)
print(as_text, type(as_text))

back_to_dict = json.loads(as_text)
print(back_to_dict["name"], type(back_to_dict))

A real model API response arrives as JSON text over the network. Parsing it is
the same skill applied to something you did not write yourself.

In [ ]:
raw_response = '''
{
  "model": "claude-sonnet-4-6",
  "content": [{"type": "text", "text": "The capital of France is Paris."}],
  "usage": {"input_tokens": 12, "output_tokens": 8}
}
'''

data = json.loads(raw_response)

print(data["content"][0]["text"])
print(data["usage"]["output_tokens"])

In [ ]:
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
]

with open(WORK / "conversation.json", "w") as f:
    json.dump(conversation, f, indent=2)

with open(WORK / "conversation.json", "r") as f:
    loaded = json.load(f)

print(loaded[0]["content"])
print("turns saved:", len(loaded))

---

### Your turn 2

Save a dictionary of resource tags to a JSON file and read one value back. Note carefully which of the four json functions each step needs.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
tags = {"env": "prod", "owner": "platform-team", "cost-center": "CC-1180"}

with open(WORK / "tags.json", "w") as f:
    json.dump(tags, f, indent=2)

with open(WORK / "tags.json", "r") as f:
    restored = json.load(f)

print(restored["owner"])

## Finding a .env file with pathlib

`pathlib` is another built in module, this time for working with paths. This
pattern searches a few likely folders for a config file, which is exactly how
AI projects locate API keys stored outside the code.

In [ ]:
from pathlib import Path

here = Path.cwd().resolve()
loaded_from = None

for candidate in [here / ".env", here.parent / ".env", here.parent.parent / ".env"]:
    if candidate.is_file():
        loaded_from = candidate
        break

print("searched from:", here)
print("would load .env from:", loaded_from)

---

## Lab: A server inventory round trip


Read `../data/servers.txt`, which holds one server name per line.

Turn it into a list of dictionaries, where each entry has a `name`, a `region`
derived from the part of the name after the last hyphen, and a `status` of
`"unknown"`.

Save that list to `scratch/inventory.json` with an indent of 2, then read it
back from disk into a fresh variable and print how many servers you recovered
and the name of the last one.

Finish by appending a single audit line to `scratch/audit.log` recording how
many servers were processed. Run the whole lab twice and confirm the audit log
has two lines while the inventory JSON still has the right count.


**Done when:**

- [ ] The text file is read with a with statement, not open/close
- [ ] Each line becomes a dictionary in a list
- [ ] json.dump writes the file and json.load reads it back
- [ ] The audit log uses append mode and grows on a second run

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

Work through these on your own after the lab. They come straight from the
course reference guide, so the wording matches what you will see there.

1. Write a script that reads a list of server names from a text file, one per line, and prints each one.
2. Save a dictionary of cloud resource tags to a JSON file, then read it back and print one of the tag values.
3. Append a new incident summary line to a running incident log file each time the script runs.
4. Save a short conversation history (a list of role/content dictionaries) to a .json file, then reload it and print the last message.

---

*Utrains &middot; support@utrains.org &middot; https://utrains.org*